# **Perineuronal net morphology (PNN) morphology analysis notebook**

<ins>**Author:**</ins> Shannon Rhoads (Github @shanrhoads)

<ins>**Notebook version:**</ins> v1.1

<ins>**Purpose:**</ins> The purpose of this notebook is to facilitate a pipeline for quantitative analysis of perineuronal net (PNN) morphology from high resolution fluorescence microscopy images. The images this pipeline was optimized for were 3D (XYZ) STED micrographs taken with a Leica STELLARIS 8 FALLCON STED microscope. Neurons from mouse tissue were stained with WFA to detect the PNN component N-acetylgalactosamine.

# <mark> TO-DO:
1. <mark> make sure there are output steps for seg and skel in the individual steps section
2. <mark> add data double checking before analysis - from infer-subc v2
3. <mark> update saving of quant to take in info about the file location from the folders

## **Getting started:**
1. Clone this repository to your local or remote computer where this notebook will be run:
    - In your computers terminal, naviate to the location you wish to store the clone of the repository
    - Execute the following code in your computer's terminal:
        > ``` Python
        > git clone https://github.com/shanrhoads/PNN-morpho-quant.git
        > ```
2. Install the following packages into your computer's base environment or into a new Python environment. Execute the following code line by line in your computers terminal (e.g., Command Prompt for Windows)
    - If creating a conda environment for this project to manage your packages
        - Install [Anaconda](https://www.anaconda.com/download) and add to PATH variables during setup if running on a Windows computer.
        - Run the following commands in your terminal:
            > ```Python
            > conda create -n PNN-morpho python=3.13
            > y
            > conda activate PNN-morpho
            ```
        - Continue with the non-conda environment instructions below
    - If you are installing packages into your computer's based environment
        - Run the following commands in your terminal:
            > ```python
            > pip install python==3.13
            > ```
    - After creating a conda environment and/or install python
        - Run the following commands in your terminal:
            > ```Python
            > pip install ipython ipykernel
            > pip install bioio bioio-ome-tiff bioio-tifffile
            > pip install napari[all] # sometimes this presents people with some errors; there are some alternative install syntax to try if so
            > pip intall dask-image # only is using dask image reading
            > pip install skan
            > pip install napari-ome-zarr # only if using zarr formatting
            > pip install bioio-lif # reading the raw .lif files
            > ```
2. Download [Visual Studio Code](https://code.visualstudio.com/Download) 
3. Open this file in VSCode
4. Click on `Select Kernel` in the top right; choose `Python Environment...` and either `PNN-morpho` or `base` from the list of options based on your choice above.



## **Notebook organization:**

This notebook includes the following sections that should be run in order for each dataset:
1. **Imports** - this section imports the necessary Python packages and function to run the analysis pipeline below (required each run).
2. **Testing Analysis Settings (on select single images)** - This section explains the individual steps included in the final segmentation, skeletonization, and quantification batch process functions. When beginning the analysis for a new, independent dataset, utilize this section to optimize the segmentation and skeletonization parameters before batch processing. It is recommended to test the selected settings on a few images across biological replicates (if possible) and experimental conditions to increase the chances of choosing settings that will be broadly applicable for your data.
3. **Batch Process segmentation and skeletonization (all images in one data folder)** - This section allows you to apply your chosen settings to a set of images from a single folder. The segmentation and skeletonization output files will be saved in a separate specified location. This section is intended to be run separately for data from each biological replicate (contained in one folder).
4. **Batch Process morphological quantification (all images in one data folder)** - Once the segmentation and skeletonization has been batch processed and the outputs are visually inspected for accuracy, the data from one biological replicate can be quantified. The inputs include the raw intensity image used for segmentation/skeletonization and the segmentation/skeletonization files.
5. **Summarize quantitative data per image (quantitative data from multiple folders)** - The quantitative data is then summarized per cells across all images in the dataset. The input is intended to include a list of file paths to all of the quantitative data that will be included during statistical analysis (all biological replicates), though is can also be applied to a single folder of data if only one is listed in the input.

## **Recommended data organization:**

It is recommended to maintain the following file structure:

``` bash
experiment-name-1/
├── Male/
|   ├── WT/                                                                         # input for steps 3 & 4
|   |   └── ?_?_cell_cell-num_region_subject-ID_stain_decon_0.tif                   # raw input file
|   ├── cKO/                                                                        # input for steps 3 & 4
|   |   └── ?_?_cell_cell-num_region_subject-ID_stain_decon_0.tif                   # raw input file
|   ├── processing-data_WT-seg-skel/                                                # result of step 3; input for step 4
|   |   └── ?_?_cell_cell-num_region_subject-ID_stain_decon_0-PNN_instance_seg.tif  # instance segmentation of the PNN
|   |   └── ?_?_cell_cell-num_region_subject-ID_stain_decon_0-PNN_skeleton.tif      # skeleton of the instance segmentation
|   ├── processing-date_cKO-seg-skel/                                               # result of step 3; input for step 4
|   |   └── ?_?_cell_cell-num_region_subject-ID_stain_decon_0-PNN_instance_seg.tif  # instance segmentation of the PNN
|   |   └── ?_?_cell_cell-num_region_subject-ID_stain_decon_0-PNN_skeleton.tif      # skeleton of the instance segmentation
|   ├── processing-date_WT-quant/                                                   # result of step 4; input for step 5
|   |   └── <mark> SOMETHING HERE                                                   # output quantification (one row of data per PNN piece quantified)
|   └── processing-date_cKO-quant/                                                  # result of step 4; input for step 5
|       └──  <mark> SOMETHING HERE                                                  # output quantification (one row of data per PNN piece quantified)
├── Female/
|   └── ...
├── Summary-quantification/                                                         # result of step 5
└   └── summary-stats.csv                                                           # per image summary statistics (one row of data per image)
experiment-name-2/
```

__________
## **1. Imports:**
Below, the packages/functions necessary for this analysis are imported.

In [2]:
# IMPORTS
from pathlib import Path
from typing import Union #,Tuple, Any
import time

from bioio import BioImage
# from bioio.writers import OmeTiffWriter
# from tifffile import imwrite

import napari

# from dask_image.imread import imread
import numpy as np
import skimage
import skan
import matplotlib.pyplot as plt
from scipy import stats
# import infer_subc

import sys
sys.path.insert(0, str(Path("..").resolve()))
from src.image_processing import skeletonize_plus, batch_PNN_seg_skel
from src.quantification import surface_area_from_props, batch_PNN_quant

import pandas as pd
pd.set_option('display.max_columns', None)

----------

## **2. Testing Analysis Settings (on select single images):**

### **1. Read file and metadata**

#### **1A. List files in path**

Specify the following information:
- `file_path`: string of the file path where the input images are located
- `file_type`: string of the input file type (e.g., ".tif") 

Then run the cell below to read in the list of file of that type found in the location specified.

In [ ]:
### USER INPUTS ###
file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO"  # OPTIONS:  WT" cKO"
file_type = ".tif"



### PROCESSING - no edits below ###
# open a Napari viewer window to visualize images & processing steps
viewer = napari.Viewer()

# create sorted list of files in directory with specified file type
file_list = sorted(Path(file_path).glob(f"*{file_type}"))

# print list of files with associated index number for selection below
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":file_list})

Specify the following information:
- `file_index`: the index of the image you would like to look at in the analysis below. The index value is found to the left of the file paths listed in the table above.

Then run the cell below to read in and view some of its metadata.

#### **1B. Select file of interest for testing**

In [ ]:
### USER INPUTS ###
file_index = 2  # change this number to select a different file from the list above



### PROCESSING - no edits below ###
# read in file
raw_file = BioImage(str(file_list[file_index]))
raw_img = raw_file.data
metadata = raw_file.standard_metadata

# save relevant metadata information as objects for use later
voxel_size_ZYX = (raw_file.physical_pixel_sizes.Z, raw_file.physical_pixel_sizes.Y, raw_file.physical_pixel_sizes.X)

# print relevant info about the image
print("File shape:", raw_file.shape)
print("Dimensions:", raw_file.dims)
print("Image channels:", raw_file.channel_names)
print("Voxel size:", raw_file.physical_pixel_sizes)


### ALTERNATIVE - if .lif file/metadata desired ###
# # read in .lif file
# lif_path = r"W:\Baldwin Lab\Hayli Spence-Osorio\PNN Project Data\Pair 4\Metadata\250429_HSO_PNN_Pair_4_Day_1.lif"
# lif_imgs = BioImage(lif_path)

# # extract raw image data and metadata
# lif_img_raw = lif_imgs.data
# lif_metadata = lif_imgs.metadata

# # view image in napari
# viewer.add_image(lif_img_raw, scale=voxel_size_ZYX, name="Deconvolved PNN image")

#####################################################################
### DEPRICATED ###
# # alternative read approach using dask array
# # the dask array approach before is formatted to utilize a series of tif images that separate channels, z's, time, etc.. 
# # Zarr formatting may still be the best approach for single files
# stack = imread(str(file_list[file_index]))
# napari.imshow(stack, multiscale=False)

#### **4C. [*optional*] Select small subregion in image to speedup processing below**

Specify the following:
- `use_small_region`: True/False to indicate if you would like to only process a smaller portion of the image (memory saving step) in the steps below, or not. *Note: if you would like to change which chunk of the image you are selecting, you can adjust the coordinates being selected within the np.squeeze()[] function's square brackets.* 

Then run the cell below to select the small portion of your image, or the entire image. The output can be visualized in the Napari window.

In [ ]:
### USER INPUTS ###
use_small_region = True  # set to True to use a small region of the image for testing purposes 


### PROCESSING - no edits below ###
# select small region or full image for testing analysis
if use_small_region:
    test_img = np.squeeze(raw_img)[30:45, 800:1250, 600:1050]
else:
    test_img = np.squeeze(raw_img)

# visualize image in napari
viewer.layers.clear()
viewer.add_image(test_img, scale=voxel_size_ZYX, name="Deconvolved PNN image")
print("The test image has been added to the Napari viewer.")

### **2. Segment PNN**

#### **2A. Rescale intensity values**


No user input is required.

The cell below rescales the intensity values per image so the max value is always 1 and the minimum value is always. This should help normalize the segmentation outcomes across images if the value ranges vary by image.

<mark> function?

In [ ]:
### PROCESSING - no edits below ###
# calculate min/max values
strech_min = test_img.min()
strech_max = test_img.max()

# rescale image
rescale = (test_img - strech_min + 1e-8) / (strech_max - strech_min + 1e-8)

#### ~~**2A. Background subtraction**~~ <mark> **NOT USED AS IT TAKES A TON OF TIME TO PROCESS** - will try without this first </mark>

~~[`Rolling ball background subtraction`](https://scikit-image.org/docs/0.25.x/auto_examples/segmentation/plot_rolling_ball.html) can be used to remove non-uniform background from images before segmentaiton or intensity measurements. In this process the amount of background per pixel/voxel is calculated from a region about the image (here defined as the radius). The radius (in voxels) can be adjusted to modify the effects of the rolling ball algorithm; it should be larger than the largest object/structure of interest in your image.~~



In [ ]:
# ### USER INPUT ###
# bg_radius = 50

# # calculate background per pixel using the rolling ball method
# bg = skimage.restoration.rolling_ball(test_img, radius=bg_radius)
# bg_subtract = test_img - bg

# # visualize output
# viewer.add_image(bg_subtract, scale=voxel_size_ZYX, name="Background subtracted")

#### ~~**2A. Denoising**~~ <mark> **NOT USED AS IT TAKES A TON OF TIME TO PROCESS** - will try without this first </mark>

~~There is quite a bit of speckley noise in your image that is making segmentation of the PNN intensity more difficult. Below, [`skimage.restoration`](https://scikit-image.org/docs/0.25.x/api/skimage.restoration.html#skimage.restoration.denoise_bilateral) module is used to denoise the image.~~

In [ ]:
# denoised = skimage.restoration.denoise_nl_means(test_img, patch_size=50, patch_distance=100)

# viewer.add_image(denoised, scale=voxel_size_ZYX, name="Denoised")

#### **2B. Smoothing**

[`Gaussian`](https://scikit-image.org/docs/0.25.x/api/skimage.filters.html#skimage.filters.gaussian) and [`median`](https://scikit-image.org/docs/0.25.x/api/skimage.filters.rank.html#skimage.filters.rank.median) smoothing filters are commonly used to smooth images and reduce certain types of noise, liek the high salt and pepper noise I think is present in your WFA stained images. The filters smooth the image different ways and are commonly used in combination. I've included both options here with sigma/size filter values that can be used to adjust how much smoothing occurs (large values = more smoothing).

Specify the following:
- `gaus_sigma`: the sigma value used for gaussian smoothing. The higher the number, the more the image is smoothed
- `med_size`: the size value used for the median smoothing filter. The higher the number, the more the image is smoothed

Then run the cell below to apply the smoothing filters. The result can be visualized in the Napari window.

<mark> function?

In [ ]:
### USER INPUT ###
gaus_sigma = 2
med_size = 8



### PROCESSING - no edits below ###
# applying smoothing filters
if gaus_sigma:
    smoothed = skimage.filters.gaussian(test_img, sigma=gaus_sigma)
else:
    smoothed = test_img

if med_size:
    fp = skimage.morphology.footprint_rectangle((round(med_size*(voxel_size_ZYX[0]/float(np.max(voxel_size_ZYX)))), 
                                                 round(med_size*(voxel_size_ZYX[1]/float(np.max(voxel_size_ZYX)))), 
                                                 round(med_size*(voxel_size_ZYX[2]/float(np.max(voxel_size_ZYX))))))
    smoothed = skimage.filters.median(smoothed, footprint=fp)

# visualize outputs
viewer.add_image(smoothed, scale=voxel_size_ZYX, name=f"Smoothed: gaus={gaus_sigma}, med={med_size}")

#### **2C. Thresholding** (multiple options below; choose 1)

There any many types of thresholding methods available in Python. The simplest form is to apply a manual threshold cutoff value to the image (all pixels/voxels with this intensity value and above will be included in the segmentation). Alternatively, the [`skimage`](https://scikit-image.org/docs/stable/api/skimage.filters.html) package has many mathematical approaches to calculate the approate threshold value based on the intensity values in the image within its `threshold` module. These automated threshold can help to adjust segmentation outcomes based on image-to-image variations. 

Both manual and automated thresholding approaches can be applied to the entire image (globally). Automated thresholds can also be applied in a local or adaptive fashion where a small local region surround each voxel area used to set a specific threshold value per voxel. Local thresholding can help to adjust the segmentation outcomes within an image if there are intensity or background variations in different regions.

<ins>**Approach 1:**</ins> Manual, global thresholding

In this approach, a cutoff value, representing the minimum intensity value to include as part of the segmentation, is specified by the user. Any voxels with an intensity value less than the cutoff will be excluded from the semantic segmentation.

Specify the following:
- `manual_cutoff`: The minimum intensity value you wish to include in your segmentation

Then run the cell below to apply the cutoff value to your entire image. The resulting segmentation is output into the Napari window.

In [ ]:
### USER INPUT ###
manual_cutoff = 0.045


### PROCESSING - no edits below ###
# select everything above threshold value for segmentation
seg = smoothed >= manual_cutoff

# visualize segmentation
viewer.add_image(seg, scale=voxel_size_ZYX, name=f"Segmentation: thresh={manual_cutoff}", blending="additive", opacity=0.4, colormap='magenta')

<ins>**Approach 2:**</ins> Automated thresholding

In this approach, the threshold cutoff value per image is calculated based on the range of intensity values within that image. The common automated thresholding approaches included in the ['skimage'](https://scikit-image.org/docs/stable/api/skimage.filters.html) package can be applied here.

Specify the following:
- `automated_method`: the name of the automated thresholding method. Options include: 'otsu', 'li', 'yen', 'isodata', 'triangle', 'minimum', 'mean'. You can read more about each method here: ['skimage.filters` documentation](https://scikit-image.org/docs/0.23.x/api/skimage.filters.html#module-skimage.filters) and [thresholding examples](https://scikit-image.org/docs/0.23.x/auto_examples/segmentation/plot_thresholding.html)

Then run the cell below to apply your chosen thresholding approach to the entire image. The resulting semgnetation is output in the Napari window.

In [ ]:
### USER INPUT ###
threshold_method = 'multiotsu'      # OPTIONS: 'otsu', 'li', 'yen', 'isodata', 'triangle', 'minimum', 'mean', 'multiotsu'
adjust = 0.55                      # OPTIONS: 1 = no threshold adjustment, <1 = more stuff selected, >1 = less stuff selected
multiotsu_middle_to = 'background'  # OPTIONS: 'foreground', 'background'


### PROCESSING - no edits below ###
# Apply automated threshold based on selected method
if threshold_method == 'otsu':
    thresh_value = skimage.filters.threshold_otsu(smoothed)
elif threshold_method == 'multiotsu':
    thresholds = skimage.filters.threshold_multiotsu(smoothed, classes=3)
    if multiotsu_middle_to == 'foreground':
        thresh_value = thresholds[0]  # select the second highest threshold
    elif multiotsu_middle_to == 'background':
        thresh_value = thresholds[1]   # select the lowest threshold
    else:
        raise ValueError(f"Unrecognized multiotsu middle to option: {multiotsu_middle_to}")
elif threshold_method == 'li':
    thresh_value = skimage.filters.threshold_li(smoothed)
elif threshold_method == 'yen':
    thresh_value = skimage.filters.threshold_yen(smoothed)
elif threshold_method == 'isodata':
    thresh_value = skimage.filters.threshold_isodata(smoothed)
elif threshold_method == 'triangle':
    thresh_value = skimage.filters.threshold_triangle(smoothed)
elif threshold_method == 'minimum':
    thresh_value = skimage.filters.threshold_minimum(smoothed)
elif threshold_method == 'mean':
    thresh_value = skimage.filters.threshold_mean(smoothed)
else:
    raise ValueError(f"Unrecognized threshold method: {threshold_method}")

# Apply threshold with optional adjustment
seg_auto = smoothed >= thresh_value*adjust

# Print the calculated threshold value for reference
print(f"Calculated threshold value using {threshold_method}: {thresh_value*adjust}")

# Visualize segmentation
viewer.add_image(seg_auto, scale=voxel_size_ZYX, name=f"Auto seg: {threshold_method} (thresh={thresh_value*adjust})", blending="additive", opacity=0.4, colormap='green')


# ### FOR TESTING ALL THE METHODS ###
# ### Compare multiple automated threshold methods ###
# # the following script can be used to visualize the different thresholding methods available in skimage
# methods = ['otsu', 'li', 'yen', 'isodata', 'triangle', 'minimum', 'mean', 'multiotsu']
# multiotsu_middle = 'foreground'  # OPTIONS: 'foreground', 'background'

# for method in methods:
#     if method == 'otsu':
#             thresh_val = skimage.filters.threshold_otsu(smoothed)
#     elif method == 'multiotsu':
#         thresholds = skimage.filters.threshold_multiotsu(smoothed, classes=3)
#         if multiotsu_middle == 'foreground':
#             thresh_val = thresholds[0]  # select the second highest threshold
#         elif multiotsu_middle == 'background':
#             thresh_val = thresholds[1]   # select the lowest threshold
#         else:
#             raise ValueError(f"Unrecognized multiotsu middle to option: {multiotsu_middle}")
#     elif method == 'li':
#         thresh_val = skimage.filters.threshold_li(smoothed)
#     elif method == 'yen':
#         thresh_val = skimage.filters.threshold_yen(smoothed)
#     elif method == 'isodata':
#         thresh_val = skimage.filters.threshold_isodata(smoothed)
#     elif method == 'triangle':
#         thresh_val = skimage.filters.threshold_triangle(smoothed)
#     elif method == 'minimum':
#         thresh_val = skimage.filters.threshold_minimum(smoothed)
#     elif method == 'mean':
#         thresh_val = skimage.filters.threshold_mean(smoothed)
#     else:
#         raise ValueError(f"Unrecognized threshold method: {method}")
    
#     seg_temp = smoothed >= thresh_val*adjust
#     print(f"{method}: threshold = {thresh_val*adjust}")
#     viewer.add_image(seg_temp, scale=voxel_size_ZYX, name=f"{method} ({thresh_val*adjust})", blending="additive", opacity=0.4, colormap='green')

<ins>**Approach 3:**</ins> Local, automated thresholding 

Above, the threshold was applied globally (to the whole image). Below, we will apply the threshold locally. This determines the threshold value for each pixel/voxel in the image based on the intensity values in a local region around it. The region size and be modified as needed to adjust the thresholding outcomes. The approach could improve segmentation in areas where the intensity range or amount of background varies across different regions within the same image.

*Note: <ins>this approach is MUCH SLOWER</ins> for each image as it has to calculate the threshold cutoff for each voxel separately.*

In [ ]:
smoothed.max()

In [ ]:
# rescale and convert to 8-bit for local thresholding
max_val = smoothed.max()
smoothed_8bit = np.round((smoothed / max_val)*255).astype(np.uint8)

In [ ]:
smoothed_8bit_downscaled = (smoothed_8bit // 2).astype(np.uint8)

np.unique(smoothed_8bit_downscaled)

In [ ]:
### USER INPUT ###
# determine the method to use for local thresholding
local_method = 'otsu'       # OPTIONS: 'gaussian', 'mean', 'median', 'otsu', 'li'
local_size = 71            # must be odd; large sizes (takes MORE time) for larger structures, small sizes for smaller structures

# Some methodsrelevant parameters
guassian_sigma_local = 40    # only used if local_method is 'gaussian'


### PROCESSING - no edits below ###
# Calculate global automated threshold value
if local_method == 'otsu':
    # create footprint for local region
    footprint = skimage.morphology.ball(local_size)
    local_otsu_threshold = skimage.filters.rank.otsu(smoothed_8bit_downscaled, footprint)
    seg_local_auto = smoothed_8bit_downscaled > local_otsu_threshold
elif local_method == 'mean':
    local_mean_threshold = skimage.filters.threshold_local(smoothed_8bit, block_size=local_size, method='mean')
    seg_local_auto = smoothed_8bit >= local_mean_threshold
### MEDIAN LOCAL THRESHOLD NOT WORKING ###
# elif local_method == 'median':
#     local_median_threshold = skimage.filters.threshold_local(smoothed, block_size=local_size, offset=0.1, method='median')
#     seg_local_auto = smoothed >= local_median_threshold
elif local_method == 'gaussian':
    local_gaussian_threshold = skimage.filters.threshold_local(smoothed_8bit, block_size=local_size, method='gaussian', param=guassian_sigma_local)
    seg_local_auto = smoothed_8bit >= local_gaussian_threshold
### NOT TESTED YET ###
# elif local_method in ['li', 'yen']:
#     if local_method == 'li':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_li(neighborhood)
#             return funct
#     elif local_method == 'yen':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_yen(neighborhood)
#             return funct
#     elif local_method == 'isodata':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_isodata(neighborhood)
#             return funct
#     elif local_method == 'triangle':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_triangle(neighborhood)
#             return funct
#     elif local_method == 'minimum':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_minimum(neighborhood)
#             return funct
    
#     local_thresholds = skimage.filters.threshold_local(smoothed, block_size=local_size, method='generic', param=thresh_method)
#     seg_local_auto = smoothed >= local_thresholds
else:
    raise ValueError(f"Unrecognized local threshold method: {local_method}")

viewer.add_image(seg_local_auto, scale=voxel_size_ZYX, name=f"Local {local_method.capitalize()} segmentation", blending="additive", opacity=0.4, colormap='cyan')

In [ ]:
### FOR REFERENCE - this works to process local otsu ###
# local_size = 100      ## For [15,450,450] sized image: size=100 -> ~9mins (pretty good outcome); size=11 much faster (created too many smaller objs); size=50 --> 2 mins, 3 sec (similar outcome as manual seg)

# smoothed_uint8 = (smoothed / smoothed.max() * 255).astype(np.uint8)

# footprint = skimage.morphology.ball(local_size)

# local_otsu_threshold = skimage.filters.rank.otsu(smoothed_uint8, footprint)
# binary_local_otsu = smoothed_uint8 > local_otsu_threshold

# viewer.add_image(smoothed_uint8, scale=voxel_size_ZYX, name=f"uint8 smoothed image")
# viewer.add_image(local_otsu_threshold, scale=voxel_size_ZYX, name=f"Local Otsu threshold (radius={local_size})")
# viewer.add_image(binary_local_otsu, scale=voxel_size_ZYX, name=f"Local Otsu segmentation (radius={local_size})", blending="additive", opacity=0.4, colormap='cyan')

#### **2D. Clean-up segmentation**

Segmentations can have errors due to imperfections in the thresholding output. Two different refining setups have been added below: removing small objects and filling small holes.

#### Remove small objects:

In [ ]:
### USER INPUTS ###
obj_min_diameter = 10
obj_method = '3D' # OPTIONS: 'slices' or '3D'
seg = seg_local_auto  # choose which segmentation to use for object filtering; OPTIONS: seg, seg_auto, seg_local_auto


### PROCESSING - no edits below ###
# filter objects based on size
if obj_method == 'slices':
    filtered = np.zeros_like(seg)
    for z in range(seg.shape[0]):
        input = seg[z,:,:]
        seg_size_filter = skimage.morphology.remove_small_objects(input, min_size=obj_min_diameter**2)
        input = np.expand_dims(seg_size_filter, axis=0)
        filtered[z,:,:] = input
elif obj_method == '3D':
    filtered = skimage.morphology.remove_small_objects(seg, min_size=obj_min_diameter**3)
else:
    SyntaxError("Unrecognized method chosen. Options include: 'slices' or '3D'.")

# visualize
viewer.add_image(filtered, scale=voxel_size_ZYX, name=f"Filter obj: method={obj_method}, obj={obj_min_diameter}", blending="additive", opacity=0.4, colormap='cyan')

#### Fill small holes

In [ ]:
### USER INPUTS ###
small_hole_diameter_max = 0
hole_method = 'slices' # OPTIONS: 'slices' or '3D'


### PROCESSING - no edits below ###
# fill holes based on size
if hole_method == 'slices':
    filled = np.zeros_like(filtered)
    for z in range(filtered.shape[0]):
        input = filtered[z,:,:]
        seg_fill_holes = skimage.morphology.remove_small_holes(input, small_hole_diameter_max**2, connectivity=8)
        input = np.expand_dims(seg_fill_holes, axis=0)
        filled[z,:,:] = input
elif hole_method == '3D':
    filled = skimage.morphology.remove_small_holes(filtered, small_hole_diameter_max**3, connectivity=26)
else:
    SyntaxError("Unrecognized method chosen. Options include: 'slices' or '3D'.")

# visualize
viewer.add_image(filled, scale=voxel_size_ZYX, name=f"Fill holes: method={hole_method}, hole={small_hole_diameter_max}", blending="additive", opacity=0.4, colormap='cyan')

#### **2E. Create instance segmentation**

The instance segmentation will not be used for skeletonization, but will be able to tell us how many separate pieces of the PNN exist.

In [ ]:
### PROCESSING - no edits below ###
# create instance seg
instance_seg = skimage.morphology.label(filled)

# visualize output
viewer.add_labels(instance_seg, scale=voxel_size_ZYX, name=f"Instance segmentation", opacity=0.4)

### **3. Skeletonize segmentation**

[Skeletonization](https://scikit-image.org/docs/0.25.x/auto_examples/edges/plot_skeleton.html) is the process by which a 2D or 3D object is narrowed to a pixel-wide representation of the original area/volume. Then, the skeleton is converted into a network graph using the [`skan`](https://skeleton-analysis.org/stable/) package for easier downstream manipulations.

The napari visualization in the next step shows the skeleton branches color coded by length.

#### **3A. Create skeleton & summarize information about each individual branch**

The summary table includes the following information:
- `skeleton_id`: Unique ID for each separate skeleton object (derived from the instance segmentation label ID)
- `branch_id`: Sequential unique ID for each branch/path in the skeleton (0 to n_paths-1)
- `random_branch_id`: Randomly permuted branch ID for visualization purposes
- `node_id_src`: Pixel index ID of the source/starting node of the branch
- `node_id_dst`: Pixel index ID of the destination/ending node of the branch
- `branch_distance`: Total distance along the branch path in physical units (µm), accounting for pixel spacing
- `branch_type`: Classification of the branch topology:
    - 0 = endpoint-to-endpoint (isolated branch)
    - 1 = junction-to-endpoint
    - 2 = junction-to-junction
    - 3 = isolated cycle
- `mean_pixel_value`: Mean intensity value of pixels along the branch path
- `stdev_pixel_value`: Standard deviation of intensity values along the branch path
- `image_coord_src_0`, `image_coord_src_1`, `image_coord_src_2`: Source node coordinates in image space (pixels) for Z, Y, X respectively
- `image_coord_dst_0`, `image_coord_dst_1`, `image_coord_dst_2`: Destination node coordinates in image space (pixels) for Z, Y, X respectively
- `coord_src_0`, `coord_src_1`, `coord_src_2`: Source node coordinates in physical space (µm) for Z, Y, X respectively
- `coord_dst_0`, `coord_dst_1`, `coord_dst_2`: Destination node coordinates in physical space (µm) for Z, Y, X respectively
- `euclidean_distance`: Straight-line (Euclidean) distance between source and destination nodes in physical units (µm)

In [ ]:
### PROCESSING - no edits below ###
# create skeleton from the instance segmentation
labeled_skel, skeleton = skeletonize_plus(instance_seg)

# convert to network graph based  labeled skeleton object
skel_g = skan.Skeleton(labeled_skel, spacing=voxel_size_ZYX, value_is_height=False)

# visualize output
viewer.layers.clear()
viewer.add_image(smoothed, scale=voxel_size_ZYX, name=f"Smoothed Input")
viewer.add_image(filled, scale=voxel_size_ZYX, name=f"Segmentation", blending="additive", opacity=0.3)
viewer.add_labels(instance_seg, scale=voxel_size_ZYX, name=f"Instance segmentation", blending="additive", opacity=0.8)

all_paths = [skel_g.path_coordinates(i) for i in range(skel_g.n_paths)]
paths_table = skan.summarize(skel_g, separator='_')
paths_table.insert(1, 'branch_id', np.arange(skel_g.n_paths))
paths_table.insert(2, 'random_branch_id', np.random.default_rng().permutation(skel_g.n_paths))

# replace skeleton_id with the ID of the organelle object from which the skeleton branch originated
if not np.any(skel_g.path_stdev()):
    # checker to see if all path points and nodes come from the same object
    paths_table['skeleton_id'] = skel_g.path_means().astype(int)
else:
    raise ValueError("at least one branch came from different organelle objects")

# calculate the degree of connectivity for each branch end point
endpoints_src = skel_g.paths.indices[skel_g.paths.indptr[:-1]]
endpoints_dst = skel_g.paths.indices[skel_g.paths.indptr[1:] - 1]

deg_src = skel_g.degrees[endpoints_src]
deg_dst = skel_g.degrees[endpoints_dst]
paths_table['deg_src'] = deg_src
paths_table['deg_dst'] = deg_dst

# view skeleton and paths table
viewer.add_shapes(all_paths, shape_type='path', properties=paths_table, edge_width=1, edge_color='skeleton_id', edge_colormap='hsv', scale=voxel_size_ZYX, opacity=1, name="Skeleton")
paths_table.set_index(['skeleton_id', 'branch_id']).sort_index()

#### **3B. Refine skeleton**
Based on measures calculated by the `skan` package, the skeleton can be refined.

Here, I've chosen to refine the skeleton by removing the shortest branches, but any of the skeleton metrics included in the table below could be used to refine the skeleton.

In [ ]:
### PROCESSING - no edits below ###
# plot the histogram of branch lengths using plt.hist from matplotlib package
b_len = paths_table['branch_distance']
possible_range = (b_len.min(), b_len.max())
num_bins = round(possible_range[1]-possible_range[0])

plt.hist(paths_table['branch_distance'], bins=num_bins*50, range=possible_range, density=False, alpha=0.7)
plt.title('Branch Length Histogram')
plt.xlabel('Branch Length (µm)')
plt.ylabel('Number of Branches')
plt.grid(axis='y', alpha=0.75)
plt.show()

Using the histogram above and napari visualization, choose the minimum branch length you want to keep within your skeleton object

In [ ]:
### USER INPUT ###
min_branch_len = 0.3


### PROCESSING - no edits below ###
# select only branches that have end points
endpoint_indices = paths_table.index[(paths_table['deg_src'] == 1) | (paths_table['deg_dst'] == 1)]
print(f"Number of endpoint branches: {len(endpoint_indices)}")

# select branches that are the min size or below
short_paths_indices = paths_table.index[paths_table['branch_distance'] < min_branch_len]
print(f"Number of branches shorter than {min_branch_len} µm: {len(short_paths_indices)}")

# find indices that are endpoints and shorter than the determined size
indices_removed = pd.Index(set(endpoint_indices) & set(short_paths_indices))
print(f"Number of branches to be removed (endpoints and short): {len(indices_removed)}")

# remove those branches from the skeleton object & the paths table
pruned_skeleton = skel_g.prune_paths(indices_removed)
paths_table_pruned = paths_table.drop(indices_removed)

# visualize output
all_paths_pruned = [pruned_skeleton.path_coordinates(i) for i in range(pruned_skeleton.n_paths)]
viewer.add_shapes(all_paths_pruned, shape_type='path', properties=paths_table_pruned, edge_width=1, edge_color='skeleton_id', edge_colormap='hsv', scale=voxel_size_ZYX, opacity=1, name="Skeleton pruned")
paths_table_pruned.set_index(['skeleton_id', 'branch_id'], inplace=True)
paths_table_pruned.sort_index(inplace=True)
paths_table_pruned

##### **Test export of skeleton object as image (for batch processing below)**

In [ ]:
### PROCESSING - no edits below ###
#  convert skeleton to image for visualization
skel_g_image = pruned_skeleton.skeleton_image
print("Skeleton image shape/unique obj IDs:", skel_g_image.shape, np.unique(skel_g_image))

# visualize output (this should be savable as a tiff if desired)
viewer.add_labels(skel_g_image.astype(int), scale=voxel_size_ZYX, name="Pruned skeleton labels", opacity=0.6)

### **4. Measure PNN features**

#### **4A. Object size/shape and intensity measures per PNN objects and whole PNN**

Here, the instanace segmentation is quantified per PNN object and from the whole PNN (all PNN object combined into one)

In [ ]:
### PROCESSING - no edits below ###

### CONTNUING to quantification
# list properties to include in regionprops analysis
properties = ['label', 'bbox', 'centroid', 'num_pixels', 'area', 'equivalent_diameter', 
              'major_axis_length', 'minor_axis_length', 'extent', 'solidity', 'euler_number',
              'min_intensity', 'max_intensity', 'mean_intensity', 'intensity_std']

# process regionprops analysis for EACH PNN OBJECT SEPARATELY
props_obj = skimage.measure.regionprops_table(label_image=instance_seg, 
                                            intensity_image=test_img,
                                            properties=properties,
                                            spacing=voxel_size_ZYX)

props_obj_tab = pd.DataFrame(props_obj)
props_obj_tab.insert(0, 'object', 'PNN fragment')

surface_area_tab = pd.DataFrame(surface_area_from_props(instance_seg, props_obj, voxel_size_ZYX), columns=['surface_area'])
props_obj_tab.insert(13, 'surface_area', surface_area_tab['surface_area'])

# process regionprops analysis for WHOLE PNN OBJECT (one per image)
# ensure semantic segmentation only constists of 1 object (ID=1)
whole_PNN = (filled>0).astype(np.uint8)

props_whole = skimage.measure.regionprops_table(label_image=whole_PNN, 
                                                intensity_image=test_img,
                                                properties=properties,
                                                spacing=voxel_size_ZYX)
props_whole_tab = pd.DataFrame(props_whole)
props_whole_tab.insert(0, 'object', 'whole PNN')

surface_area_tab = pd.DataFrame(surface_area_from_props(whole_PNN, props_whole, voxel_size_ZYX), columns=['surface_area'])
props_whole_tab.insert(13, 'surface_area', surface_area_tab['surface_area'])


# combine both tables
combined_props_tab = pd.concat([props_obj_tab, props_whole_tab], ignore_index=True)

# rename & add columns for clarity and additional information
combined_props_tab.insert(0, 'image_name', file_list[file_index].name)
combined_props_tab.rename(columns={'area':'volume'}, inplace=True)
combined_props_tab['intensity_sum'] = combined_props_tab['mean_intensity'] * combined_props_tab['num_pixels']
combined_props_tab.insert(15, "SA_to_volume_ratio", combined_props_tab["surface_area"].div(combined_props_tab["volume"]))
rounded_scale = tuple(round(x, 2) for x in voxel_size_ZYX)
combined_props_tab.insert(1, "scale", str(rounded_scale))

display(combined_props_tab)

#### **~~4A. Intensity of WFA in PNN segmentation~~** <mark> ORIGINAL VERSION FOR WHOLE PNN ONLY - before using regionprops


In [ ]:
# ### PROCESSING - no edits below ###
# # select intensity values only where the PNN is present
# PNN_ints = test_img[filled > 0]

# # find unique PNN object IDs for counting
# unique = np.unique(instance_seg)
# unique = unique[unique != 0]

# # measure sum, mean, median, and standard deviation of intensity values
# int_dict = {"image": [str(file_list[file_index])],
#             "scale": [voxel_size_ZYX],
#             "PNN fragment count": [len(unique)],
#             "PNN total volume (voxels)": [np.count_nonzero(filled)],
#             "PNN total volume (um)": [np.count_nonzero(filled)*voxel_size_ZYX[0]*voxel_size_ZYX[1]*voxel_size_ZYX[2]],
#             "WFA total intensity (AU) in PNN": [np.sum(PNN_ints)],
#             "WFA mean intensity (AU) in PNN": [np.mean(PNN_ints)],
#             "WFA media intensity (AU) in PNN": [np.median(PNN_ints)],
#             "WFA SD intensity (AU) in PNN": [np.std(PNN_ints)]}

# PNN_int_tab = pd.DataFrame(int_dict)
# PNN_int_tab

#### **4B. Skeleton metrics**

The first table shows all the metrics that are collected using the skan package. Those metrics have been summarized per skeleton object and for all skeleton objects in the entire image.

In [ ]:
# ### PROCESSING - no edits below ###
# summarize skeleton branch table per skeleton object
paths_table_pruned.reset_index(inplace=True)
skel_sum1 = paths_table_pruned[['skeleton_id', 'branch_id']].groupby('skeleton_id').agg(['count'])
skel_sum2 = paths_table_pruned[['skeleton_id', 'branch_type']].groupby('skeleton_id').agg(['mean', 'median', 'min', 'max', 'std'])
skel_sum3 = paths_table_pruned[['skeleton_id', 'branch_distance', 'euclidean_distance']].groupby('skeleton_id').agg(['sum', 'mean', 'median', 'min', 'max', 'std'])

skel_summary = pd.concat([skel_sum1, skel_sum2, skel_sum3], axis=1)
skel_summary.columns = ['_'.join(col).strip() for col in skel_summary.columns.values]
skel_summary.reset_index(inplace=True)
skel_summary.insert(0, 'image_name', file_list[file_index].name)
skel_summary.insert(1, "scale", str(rounded_scale))
skel_summary.insert(2, 'object', 'PNN fragment')
skel_summary.rename(columns={'skeleton_id':'label',
                             'branch_id_count':'branch_count'}, inplace=True)

# summarize skeleton branch table for whole image
paths_table_pruned.insert(0, 'combined_skeleton_id', 1)  # assign all branches to skeleton ID 1 for whole image summary
combo_skel_sum1 = paths_table_pruned[['combined_skeleton_id', 'branch_id']].groupby('combined_skeleton_id').agg(['count'])
combo_skel_sum2 = paths_table_pruned[['combined_skeleton_id', 'branch_type']].groupby('combined_skeleton_id').agg(['mean', 'median', 'min', 'max', 'std'])
combo_skel_sum3 = paths_table_pruned[['combined_skeleton_id', 'branch_distance', 'euclidean_distance']].groupby('combined_skeleton_id').agg(['sum', 'mean', 'median', 'min', 'max', 'std'])

combo_skel_summary = pd.concat([combo_skel_sum1, combo_skel_sum2, combo_skel_sum3], axis=1)
combo_skel_summary.columns = ['_'.join(col).strip() for col in combo_skel_summary.columns.values]
combo_skel_summary.reset_index(inplace=True)
combo_skel_summary.insert(0, 'image_name', file_list[file_index].name)
combo_skel_summary.insert(1, "scale", str(rounded_scale))
combo_skel_summary.insert(2, 'object', 'whole PNN')
combo_skel_summary.rename(columns={'combined_skeleton_id':'label',
                                   'branch_id_count':'branch_count'}, inplace=True)

# combine both tables and format
final_skel_summary = pd.concat([skel_summary, combo_skel_summary], axis=0)

final_skel_summary

In [ ]:
### ORIGINAL CODE FROM BEFORE REGIONPROPS WAS INCLUDED - KEPT FOR REFERENCE ###

# ### PROCESSING - no edits below ###
# # skan table
# display(paths_table_pruned)

# # creating summary of skeleton metrics
# sekl_dict = {"image": [str(file_list[file_index])],
#             "scale": [voxel_size_ZYX],
#             "Total branch count": [paths_table.index.max()+1],
#             "Mean branch length (um)": [np.mean(paths_table["branch_distance"])],
#             "Median branch length (um)": [np.median(paths_table["branch_distance"])],
#             "SD branch length (um)": [np.std(paths_table["branch_distance"])]}

# skel_sum_tab = pd.DataFrame(sekl_dict)
# skel_sum_tab

#### **4C. Combine tables above for output**

In [ ]:
### PROCESSING - no edits below ###
combo = pd.merge(combined_props_tab, final_skel_summary, on= ['image_name', 'scale', 'object', 'label'], how='outer')
combo

----------

## **3/4. Batch Process segmentation, skeletonization, and quantification (all images in one data folder):**

The above code is put together into a single function that enables you to batch process all of the images in a folder.

### **Segmentation & skeletonization**

First, we will segment and skeletonize all of the cells from each folder. After segmentation is complete and checked for accuracy, the segmented images can be quantified and summarized (below). 

<ins>Processing multiple folders of data:</ins> The cell below can be copied as many times as necessary to process each folder of data sequentially. Copy and paste the cell, update the `file_path` and `out_path` information, and run the new cell.

In [2]:
batch_PNN_seg_skel(file_path="/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO",
                    file_type=".tif",
                    out_path="/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/20260309_test-final-funct_cKO-seg-skel",
                    gaus_sigma=2,
                    med_size=8,
                    manual_threshold_cutoff=0.045,
                    auto_threshold_method=None, #'multiotsu',
                    auto_threshold_adjust=None, #0.55,
                    auto_multiotsu_middle_to=None, #'background',
                    local_threshold_method=None, #'otsu',
                    local_threshold_adjust=None, #1,
                    local_threshold_size=None, #51, #71,
                    local_gaussian_sigma=None,
                    obj_min_diameter=10,
                    obj_method='3D',
                    hole_min_diameter=0,
                    hole_method='slices',
                    min_branch_len=0.3)

Found 3 .tif files in /users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO.
Processing first image:


Attempted file (/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO/382_2_Cell_One_L3_SSCX_WFA_decon_0.tif) load with reader: <class 'bioio_ome_tiff.reader.Reader'> failed with error: bioio-ome-tiff does not support the image: '/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO/382_2_Cell_One_L3_SSCX_WFA_decon_0.tif'. Failed to parse XML for the provided file. Error: not well-formed (invalid token): line 1, column 6


Applying a manual threshold of 0.045.


/users/s/r/srhoads/PNN-morpho-quant/src/image_processing.py:270: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  out_seg = skimage.morphology.remove_small_objects(src_seg, min_size=obj_min_diameter**3)
/users/s/r/srhoads/PNN-morpho-quant/src/image_processing.py:315: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imsave`.
  skimage.io.imsave(f"{out_path}/{f.stem}-PNN_instance_seg.tif", PNN_instance_seg, plugin="tiff

Processed 382_2_Cell_One_L3_SSCX_WFA_decon_0.tif in 3.2708333333333335 minutes.
Processing next image:


...

Processed 382_2_Cell_Three_L3_SSCX_WFA_decon_0.tif in 3.5513333333333335 minutes.
Processing last image:


Attempted file (/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO/382_2_Cell_Two_L3_SSCX_WFA_decon_0.tif) load with reader: <class 'bioio_ome_tiff.reader.Reader'> failed with error: bioio-ome-tiff does not support the image: '/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO/382_2_Cell_Two_L3_SSCX_WFA_decon_0.tif'. Failed to parse XML for the provided file. Error: not well-formed (invalid token): line 1, column 6


Applying a manual threshold of 0.045.


/users/s/r/srhoads/PNN-morpho-quant/src/image_processing.py:270: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  out_seg = skimage.morphology.remove_small_objects(src_seg, min_size=obj_min_diameter**3)
/users/s/r/srhoads/PNN-morpho-quant/src/image_processing.py:315: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imsave`.
  skimage.io.imsave(f"{out_path}/{f.stem}-PNN_instance_seg.tif", PNN_instance_seg, plugin="tiff

Processed 382_2_Cell_Two_L3_SSCX_WFA_decon_0.tif in 3.641833333333333 minutes.
Processed 3 images in 10.464 minutes.


### **Quality check output**

Before continue on to the quantification step, have a look at each of your images to confirm that the segmentation and skeletonization settings used created a succificient outcome. If necessary, refine your segmentation settings, or manually edit individual cells before quantification. If individual cells require editting, use step 2 above to edit the settings from that individual image. Along the way, can edit the intermediate outputs using the draw feature in Napari if refining the settings isn't accurate enough.

### **Quantification**

Now that the segmentation and skeletonization outputs hace been quality checked, we will quantify the PNN morphology features. 

<ins>Processing multiple folders of data:</ins> The cell below can be copied as many times as necessary to process each folder of data sequentially. Copy and paste the cell, update the `file_path` and `out_path` information, and run the new cell.

In [7]:
test_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO"

In [11]:
test_path.rsplit("/")[-3]

'Pair 4'

In [ ]:
def _batch_PNN_quant(raw_file_path: str,
                    raw_file_type: str,
                    seg_skel_path: str,
                    quant_out_path: str):
    """
    This function segments perineuronal nets (PNN) from 3D STED images, creates network graphs of the PNN structure, and quantifies morphological features of the PNN.

    Parameters:
    -----------
    raw_file_path: str
        location of raw image files
    raw_file_type: str
        file type of raw input images (e.g., ".tif")
    seg_skel_path: str
        location where segmentation/skeleton images are saved
    quant_out_path: str
        location where data tables will be saved; if the path does not exist, it will be created for you
    
    Output:
    -------
    summary_tab: pd.DataFrame
        Data table summarizing the quantification results for each PNN object and the entire PNN combined
        This table is automatically saved to a .csv file 

    Analysis metrics include:
    -------------------------

    """
    # confirm file paths and files exist
    if not Path.exists(Path(raw_file_path)):
        FileExistsError("Input file path does not exist.")
    elif not Path.exists(Path(seg_skel_path)):
        FileExistsError("Segmentation/skeleton file path does not exist.")
    else:
        file_list = sorted(Path(raw_file_path).glob(f"*{raw_file_type}"))
        if len(file_list) == 0:
            FileExistsError(f"Input file path does not have any {raw_file_type} files.")

    
    if not Path.exists(Path(quant_out_path)):
        Path.mkdir(Path(quant_out_path))
        print(f"Making {quant_out_path}")
    elif Path.exists(Path(quant_out_path)):
        # check if output file already exists
        if Path.exists(Path(f"{quant_out_path}/PNN_quantification.csv")):
            raise FileExistsError("Quantification output file already exists. Please choose a different quant_out_path or dataset_name to avoid overwriting.")

    # keeping track of processing time
    count=0
    start=time.time()

    # loop through list of images and process
    for f in file_list:
        img_time_start=time.time()
        count=count+1
        if count==1:
            print("Quantifying first image:")
        if count>1:
            print("Quantifying next image:")
        if count==len(file_list):
            print("Quantifying last image:")
        
        # collect paths to the related seg and skel files based on file name
        filez = {name: str(Path(seg_skel_path) / f"{f.stem}-{name}.tif") for name in ['PNN_instance_seg', 'PNN_skeleton']}


        # read intensity image
        raw_file = BioImage(str(f))
        print("imported image successfully")
        raw_image = np.squeeze(raw_file.data)

        voxel_size_ZYX = (raw_file.physical_pixel_sizes.Z, raw_file.physical_pixel_sizes.Y, raw_file.physical_pixel_sizes.X)
        rounded_scale = tuple(round(x, 4) for x in voxel_size_ZYX)

        # empty list to collect quantification tables for each image
        quant_tabs = []

        # loop through seg/skel files; read and quantify each
        for name, path in filez.items():
            if not Path.exists(Path(path)):
                raise FileExistsError(f"Expected file not found: {path}")
            
            seg = skimage.io.imread(path)
            print("imported segmentation successfully")
            
            # for segmentation files
            if name == 'PNN_instance_seg':
                # loop through both types of objects (PNN fragment and whole PNN)
                obj_quant_tabs = []
                obj_dict = {"PNN fragment": seg, 
                            "whole PNN": (seg>0).astype(np.uint8)}
                for obj_type, obj_seg in obj_dict.items():
                    print(f"quant seg: {obj_type}")
                    print("num of objs:", len(np.unique(obj_seg))-1)
                    properties = ['label', 'bbox', 'centroid', 'num_pixels', 'area', 'equivalent_diameter', 
                                'major_axis_length', 'minor_axis_length', 'extent', 'solidity', 'euler_number',
                                'min_intensity', 'max_intensity', 'mean_intensity', 'intensity_std']

                    props_obj = skimage.measure.regionprops_table(label_image=obj_seg, 
                                                                intensity_image=raw_image,
                                                                properties=properties,
                                                                spacing=voxel_size_ZYX)
                    print("finished regionprops")

                    props_obj_tab = pd.DataFrame(props_obj)
                    props_obj_tab.insert(0, 'object', obj_type)

                    surface_area_tab = pd.DataFrame(surface_area_from_props(obj_seg, props_obj, voxel_size_ZYX), columns=['surface_area'])
                    props_obj_tab.insert(13, 'surface_area', surface_area_tab['surface_area'])
                    obj_quant_tabs.append(props_obj_tab)
                    print(f"finished seg: {obj_type}")
                
                # combine both tables and format
                combined_obj_quant = pd.concat(obj_quant_tabs, ignore_index=True)
                combined_obj_quant.insert(0, 'image_name', file_list[file_index].name)
                combined_obj_quant.rename(columns={'area':'volume'}, inplace=True)
                combined_obj_quant['intensity_sum'] = combined_obj_quant['mean_intensity'] * combined_obj_quant['num_pixels']
                combined_obj_quant.insert(15, "SA_to_volume_ratio", combined_obj_quant["surface_area"].div(combined_obj_quant["volume"]))
                rounded_scale = tuple(round(x, 2) for x in voxel_size_ZYX)
                combined_obj_quant.insert(1, "scale", str(rounded_scale))

                quant_tabs.append(combined_obj_quant)

            # for skeleton files
            if name == 'PNN_skeleton':
                skel_g = skan.Skeleton(seg, spacing=voxel_size_ZYX, value_is_height=False)
                del seg  # save memory

                # create initial skeleton branch table and add additional
                paths_table = skan.summarize(skel_g, separator='_')
                paths_table.insert(1, 'branch_id', np.arange(skel_g.n_paths))
                paths_table.insert(2, 'random_branch_id', np.random.default_rng().permutation(skel_g.n_paths))
                if not np.any(skel_g.path_stdev()): 
                    paths_table['skeleton_id'] = skel_g.path_means().astype(int)
                else:
                    raise ValueError("at least one branch came from different organelle objects")
                endpoints_src = skel_g.paths.indices[skel_g.paths.indptr[:-1]]
                endpoints_dst = skel_g.paths.indices[skel_g.paths.indptr[1:] - 1]
                deg_src = skel_g.degrees[endpoints_src]
                deg_dst = skel_g.degrees[endpoints_dst]
                paths_table['deg_src'] = deg_src
                paths_table['deg_dst'] = deg_dst
                paths_table.insert(0, 'combined_skeleton_id', 1)  # assign all branches to skeleton ID 1 for whole image summary

                # loop through both types of skeletons (PNN fragment and whole PNN)
                skel_quant_tabs = []
                skel_dict = {"PNN fragment": 'skeleton_id', 
                             'whole PNN': 'combined_skeleton_id'}
                for skel_type, skel_id in skel_dict.items():
                    print(f"quant skel: {obj_type}")
                    skel_sum1 = paths_table[[skel_id, 'branch_id']].groupby(skel_id).agg(['count'])
                    skel_sum2 = paths_table[[skel_id, 'branch_type']].groupby(skel_id).agg(['mean', 'median', 'min', 'max', 'std'])
                    skel_sum3 = paths_table[[skel_id, 'branch_distance', 'euclidean_distance']].groupby(skel_id).agg(['sum', 'mean', 'median', 'min', 'max', 'std'])

                    skel_summary = pd.concat([skel_sum1, skel_sum2, skel_sum3], axis=1)
                    skel_summary.columns = ['_'.join(col).strip() for col in skel_summary.columns.values]
                    skel_summary.reset_index(inplace=True)
                    skel_summary.insert(0, 'image_name', file_list[file_index].name)
                    skel_summary.insert(1, "scale", str(rounded_scale))
                    skel_summary.insert(2, 'object', skel_type)
                    skel_summary.rename(columns={skel_id:'label',
                                                'branch_id_count':'branch_count'}, inplace=True)
                    
                    skel_quant_tabs.append(skel_summary)
                    print(f"finished skel: {obj_type}")

                # combine both skel tables
                combined_skel_quant = pd.concat(skel_quant_tabs, axis=0)

                quant_tabs.append(combined_skel_quant)

        # combine regionprops and skel tables
        combo = pd.merge(quant_tabs[0], quant_tabs[1], on= ['image_name', 'scale', 'object', 'label'], how='outer')
        print("merged together")
        display(combo)

        combo.insert(0, 'replicate', raw_file_path.rsplit("/")[-3])
        combo.insert(0, 'genotype', raw_file_path.rsplit("/")[-1])

        # write to csv
        # combo.to_csv(f"{quant_out_path}/PNN_quantification.csv", index=False, mode='a')
        # del combo  # save memory before repeating loop

        print(f"Quantified {f.name} and saved data table. Time taken: {(time.time() - img_time_start)/60} minutes.")

    print(f"Analysis finished!")
    print(f"Quantified {len(file_list)} images in {(time.time() - start)/60} minutes.")
    print("Output saved to:", f"{quant_out_path}/PNN_quantification.csv")

In [19]:
_batch_PNN_quant(raw_file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO", 
                raw_file_type = ".tif",
                seg_skel_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/20260225_test-auto-seg_cKO-seg-skel",
                quant_out_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/20260225_test-auto-seg_cKO-quant")

Attempted file (/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO/382_2_Cell_One_L3_SSCX_WFA_decon_0.tif) load with reader: <class 'bioio_ome_tiff.reader.Reader'> failed with error: bioio-ome-tiff does not support the image: '/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO/382_2_Cell_One_L3_SSCX_WFA_decon_0.tif'. Failed to parse XML for the provided file. Error: not well-formed (invalid token): line 1, column 6


Quantifying first image:
imported image successfully
imported segmentation successfully
quant seg: PNN fragment
num of objs: 159


KeyboardInterrupt: 

In [ ]:
_batch_PNN_quant(dataset_name = "20260225_test-quant", 
                raw_file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/WT", 
                raw_file_type = ".tif",
                seg_skel_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/20260225_test-auto-seg_WT-seg-skel",
                quant_out_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/20260225_test-auto-seg_WT-quant")

In [ ]:
_batch_PNN_quant(dataset_name = "20260225_test-quant", 
                raw_file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 5/3D STED/cKO", 
                raw_file_type = ".tif",
                seg_skel_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 5/3D STED/20260225_test-auto-seg_cKO-seg-skel",
                quant_out_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 5/3D STED/20260225_test-auto-seg_cKO-quant")

In [ ]:
_batch_PNN_quant(dataset_name = "20260225_test-quant", 
                raw_file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 5/3D STED/WT", 
                raw_file_type = ".tif",
                seg_skel_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 5/3D STED/20260225_test-auto-seg_WT-seg-skel",
                quant_out_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 5/3D STED/20260225_test-auto-seg_WT-quant")

In [ ]:
### USER INPUT ###
file_path = r"W:\Baldwin Lab\Hayli Spence-Osorio\PNN Project Data\Pair 5\3D STED\WT"
file_type = ".tif"

out_path = file_path+"/PNN-mopho-quant"

quant_PNN_morpho(file_path = file_path,
                 file_type = file_type,
                 out_path = out_path,
                 gaus_sigma = 2,
                 med_size = 8,
                 treshold_cutoff = 0.045, 
                 obj_max_diameter = 10,
                 obj_method = '3D',
                 small_hole_diameter_max = 0,
                 hole_method = 'slices',
                 min_branch_len = 0)

----------

## **5. Summarize quantitative data per image (quantitative data from multiple folders):**

Once all of the biological replicates have undergone quantification, the quantitative data can be summarized per image (or in this case, per cell, since each image only contains the PNN from one cell).

In [ ]:
def batch_summary_stats(csv_path_list: List[str],
                         out_path: str,
                         out_preffix: str):
    """" 
    csv_path_list: List[str],
        A list of path strings where .csv files to analyze are located.
    out_path: str,
        A path string where the summary data file will be output to
    out_preffix: str
        The prefix used to name the output file.    
    """
    ds_count = 0
    ###################
    # Read in the csv files and combine them into one of each type
    ###################
    quant_tabs = []

    for loc in csv_path_list:
        ds_count = ds_count + 1
        loc=Path(loc)
        files_store = sorted(loc.glob("*.csv"))
        for file in files_store:
            fl_count = fl_count + 1
            stem = file.stem

            if stem.endswith("-PNN_quantification"):
                test_orgs = pd.read_csv(file, index_col=0)
                test_orgs.insert(0, "dataset", stem[:-11])
                org_tabs.append(test_orgs)
            if contacts in stem:
                test_contact = pd.read_csv(file, index_col=0)
                test_contact.insert(0, "dataset", stem[:-9])
                contact_tabs.append(test_contact)
            if dist in stem:
                test_dist = pd.read_csv(file, index_col=0)
                test_dist.insert(0, "dataset", stem[:-14])
                dist_tabs.append(test_dist)
            if regions in stem:
                test_regions = pd.read_csv(file, index_col=0)
                test_regions.insert(0, "dataset", stem[:-8])
                region_tabs.append(test_regions)
            
    org_df = pd.concat(org_tabs,axis=0, join='outer')
    contacts_df = pd.concat(contact_tabs,axis=0, join='outer')
    dist_df = pd.concat(dist_tabs,axis=0, join='outer')
    regions_df = pd.concat(region_tabs,axis=0, join='outer')

    ###################
    # summary stat group
    ###################
    group_by = ['dataset', 'image_name', 'object']
    sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]
    ag_func_standard = ['mean', 'median', 'std']

    ###################
    # summarize shared measurements between org_df and contacts_df
    ###################
    org_cont_tabs = []
    for tab in [org_df, contacts_df]:
        tab1 = tab[group_by + ['volume']].groupby(group_by).agg(['count', 'sum'] + ag_func_standard)
        tab2 = tab[group_by + ['surface_area']].groupby(group_by).agg(['sum'] + ag_func_standard)
        tab3 = tab[group_by + sharedcolumns].groupby(group_by).agg(ag_func_standard)
        shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by)
        shared_metrics = pd.merge(shared_metrics, tab3, 'outer', on=group_by)
        org_cont_tabs.append(shared_metrics)

    org_summary = org_cont_tabs[0]
    contact_summary = org_cont_tabs[1]

----------
## **OLD: Visualize quant output**

In [ ]:
# read data
WT_tab = pd.read_csv(r"W:\Baldwin Lab\Hayli Spence-Osorio\PNN Project Data\Pair 4\3D STED\WT\PNN-mopho-quant\PNN-morpho-quant.csv")
cKO_tab = pd.read_csv(r"W:\Baldwin Lab\Hayli Spence-Osorio\PNN Project Data\Pair 4\3D STED\cKO\PNN-mopho-quant\PNN-morpho-quant.csv")
WT_tab2 = pd.read_csv(r"W:\Baldwin Lab\Hayli Spence-Osorio\PNN Project Data\Pair 5\3D STED\WT\PNN-mopho-quant\PNN-morpho-quant.csv")
cKO_tab2 = pd.read_csv(r"W:\Baldwin Lab\Hayli Spence-Osorio\PNN Project Data\Pair 5\3D STED\cKO\PNN-mopho-quant\PNN-morpho-quant.csv")

# combine into one tab with new "condition" column
WT_tab['condition'] = "WT"
cKO_tab['condition'] = "cKO"
WT_tab2['condition'] = "WT"
cKO_tab2['condition'] = "cKO"

WT_tab['dataset'] = "Pair 4"
cKO_tab['dataset'] = "Pair 4"
WT_tab2['dataset'] = "Pair 5"
cKO_tab2['dataset'] = "Pair 5"

combo_tab = pd.concat([WT_tab, cKO_tab, WT_tab2, cKO_tab2])
combo_tab

In [ ]:
# list quantitative columnes
quant_col = combo_tab.columns[2:-2]

# summarize the mean, standard deviation of each column
summary_tab = combo_tab.groupby(["condition"])[quant_col].agg(['mean', 'std', 'count'])

summary_tab

In [ ]:
# ### Single graph ###
# col = "Total branch count"
# conditions = list(summary_tab.index)

# pnt_data = combo_tab[["condition",col]]

# means = list(summary_tab[(col, "mean")])
# stds = list(summary_tab[(col, "std")])

# # Create the figure and axes
# fig, ax = plt.subplots()

# # Plot the bars representing the means
# bar_width = 0.6
# x_pos = np.arange(len(conditions))
# ax.bar(x_pos, means, yerr=stds, capsize=5, width=bar_width, color='skyblue', label='Mean with Std Dev')

# # Plot individual data points as scatter plots
# # Adjust x-position slightly to avoid overlap with bar center
# for i, cat in enumerate(conditions):
#     jitter = np.random.uniform(-bar_width/4, bar_width/4, int(summary_tab[(col, "count")][cat])) # Add some jitter for visibility
#     ax.scatter(x_pos[i] + jitter, list(pnt_data[pnt_data['condition']=="WT"]["Total branch count"]), color='red', s=50, alpha=0.7, label='Individual Data Points' if i == 0 else "")

# # Customize the plot
# ax.set_xticks(x_pos)
# ax.set_xticklabels(conditions)
# ax.set_ylabel('Value')
# ax.set_title('Mean Values with Standard Deviation Error Bars and Individual Data Points')
# ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
# plt.tight_layout()
# plt.show()

In [ ]:
# graph each column
conditions = list(summary_tab.index)

plots = {}
for col in quant_col:
    # find mean & SD for this value
    pnt_data = combo_tab[["condition",col]]

    means = list(summary_tab[(col, "mean")])
    stds = list(summary_tab[(col, "std")])

    # Create the figure and axes
    fig, ax = plt.subplots()

    # Plot the bars representing the means
    bar_width = 0.6
    x_pos = np.arange(len(conditions))
    ax.bar(x_pos, means, yerr=stds, capsize=5, width=bar_width, color='skyblue', label='Mean with Std Dev')

    # Plot individual data points as scatter plots
    # Adjust x-position slightly to avoid overlap with bar center
    for i, cat in enumerate(conditions):
        jitter = np.random.uniform(-bar_width/4, bar_width/4, int(summary_tab[(col, "count")][cat])) # Add some jitter for visibility
        ax.scatter(x_pos[i] + jitter, list(pnt_data[pnt_data['condition']==cat][col]), color='red', s=50, alpha=0.7, label='Individual Data Points' if i == 0 else "")

    # Customize the plot
    ax.set_xticks(x_pos)
    ax.set_xticklabels(conditions)
    ax.set_ylabel(col)
    ax.set_title(f"{col}")
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    plots[f"{col}_figure"] = fig
    plots[f"{col}_axes"] = ax

T-test may not be the right approach here, but I did implement a correction for multiple comparisons to prevent overt p-hacking hopefully.

In [ ]:
test_tab = combo_tab[["condition", "Total branch count"]]
test_tab[test_tab["condition"]=="WT"]

In [ ]:
list(combo_tab[["condition", "Total branch count"]][test_tab["condition"]=="WT"]["Total branch count"])

In [ ]:
group1 = list(combo_tab[["condition", "Total branch count"]][test_tab["condition"]=="WT"]["Total branch count"])
group2 = list(combo_tab[["condition", "Total branch count"]][test_tab["condition"]=="cKO"]["Total branch count"])

t_statistic, p_value = stats.ttest_ind(group1, group2)
p_value

In [ ]:
from scipy import stats
import numpy as np
from statsmodels.stats import multitest

# t-test
t_statistic, p_value = stats.ttest_ind(group1, group2)

# multiple comparisons
reject_bonferroni, pvals_corrected_bonferroni, _, _ = multitest.multipletests(p_value, alpha=0.05, method='bonferroni')
# OR
reject_fdr, pvals_corrected_fdr, _, _ = multitest.multipletests(p_value, alpha=0.05, method='fdr_bh')

## **Versioning Notes:**
- **V1.1**: 12/11/2025 SR updates based on feedback from HSO
    - Load file (no change)
    - Segment PNN from WFA channel
        - Thresholding: update to add local and automated thresholding methods
    - Skeletonize PNN segmentation (no change)
    - Quantify PNN morphology & marker intensity:
        - Adding per PNN object and per PNN morphology quantification (skimage regionprops) and per object/PNN intensity quant of some additional channels (if they are the same resolution)

- **V1.0**: 10/28/2025 SR creates first draft with the following goals
    - Load file
        - list files of a specific type in path
        - read files (BioIO)
    - Segment PNN from WFA channel
        - Rescale intensities: min value = 0, max value = 1
        - Background subtraction: None
        - Denoising: None
        - Smoothing: gaussian = 2, median = 8
        - Thresholding: manual (low pass filter)
        - Clean-up: remove small objects (<10, 3D), fill small holes (none)
        - Instance segmentation: connectivity-based
    - Skeletonize PNN segmentation
        - Create skeleton object (skimage & skan)
        - Refine skeleton: remove small end-point branches (<1 um)
    - Quantify PNN morphology & marker intensity:
        - Intensity within entire PNN (segmented area, not per object)
        - Count and skeleton morphology metrics (from skan)